**Naive Bayes**

Naive Bayes is a probabilistic classifier based on Bayes' Theorem. It is called "Naive" because it assumes that all features are independent of each other - an assumption that rarely holds true in real life but works surprisingly well in practice.

**Bayes theorem Formula**

$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

* $P(y|X)$: Posterior probability (Probability of a class given the data).
* $P(X|y)$: Likelihood (Probability of the data given the class).
* $P(y)$: Prior probability (How common the class is).

**Gaussian Likelihood:** Since our features are continuous, we assume they follow a Gaussian (Normal) Distribution:

$$P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

We calculate the Mean ($\mu$) and Variance ($\sigma^2$) for each feature for each class.

In [1]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
iris = datasets.load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
class GaussianNaiveBayes:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.parameters = [] 
        
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.parameters.append({
                "mean": X_c.mean(axis=0),
                "var": X_c.var(axis=0),
                "prior": X_c.shape[0] / X.shape[0]
            })

    def _calculate_likelihood(self, class_idx, x):
        mean = self.parameters[class_idx]["mean"]
        var = self.parameters[class_idx]["var"]
        
        numerator = np.exp(-((x - mean)**2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def _calculate_posterior(self, x):
        posteriors = []
        for i, c in enumerate(self.classes):
            prior = np.log(self.parameters[i]["prior"])
            
            likelihood = np.sum(np.log(self._calculate_likelihood(i, x)))
            posterior = prior + likelihood
            posteriors.append(posterior)
        return self.classes[np.argmax(posteriors)]

    def predict(self, X):
        return np.array([self._calculate_posterior(x) for x in X])

**Step 1: Calculating Priors**- The "Prior" is simply the probability of a class appearing in our training set. If 33% of the flowers are Setosa, the prior $P(Setosa) = 0.33$.

**Step 2: The Gaussian Assumption** - For each feature (like petal width), we calculate the Mean and Variance. We use these to build a Bell Curve for each class. When a new flower comes in, we check which class's Bell Curve is most likely to have produced that specific petal width.

**Step 3: The "Log" Trick** - In probability, we multiply many small decimals together. This can lead to Numerical Underflow (where the computer rounds the number to zero). To prevent this, we take the Logarithm of the probabilities and add them instead of multiplying.

In [6]:
nb = GaussianNaiveBayes()
nb.fit(X_train, y_train)
y_pred = nb.fit_transform = nb.predict(X_test)

print(f"Manual Naive Bayes Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Manual Naive Bayes Accuracy: 100.00%


In [7]:
from sklearn.naive_bayes import GaussianNB

sk_nb = GaussianNB()
sk_nb.fit(X_train, y_train)
print(f"Sklearn Accuracy: {accuracy_score(y_test, sk_nb.predict(X_test)) * 100:.2f}%")

Sklearn Accuracy: 100.00%
